# Paper author countries — `countries`, `n_countries`, first- and last-author country

The Dimensions twin of `OpenAlex/notebook/paper_author_country.ipynb`: same columns, same meanings,
so the two files can be compared paper for paper. `paper_author.parquet` here already carries the
paper-level `countries` / `n_countries`; this file adds the **author-level** view — how many authors
are located, how many authors each country contributes, and the first / last author's country.

## Source
```
/project/jevans/dimensions/dimensions/dimensions_june_2025/publications/publications_*   # 4,219 parquet files, no extension
```
Two columns are read: `id` and `authors[]`. `authors[]` is one entry per author **slot**, in author
order, each with `affiliations_address[].country_code` (ISO2). `cache/pub_authors/` cannot be reused:
it keeps only the paper-level union of countries, not each author's list.

## What an author's country is
An author slot's countries are the distinct, sorted `country_code`s over its `affiliations_address[]`
(null and empty codes dropped). A slot with none is *unlocated*. Differences from the OpenAlex twin:
- there is nothing to de-duplicate — `authors[]` is already one entry per slot, and every slot counts,
  whether or not Dimensions resolved it to a `researcher_id` (so `team_size` == `paper_author.team_size`);
- author order is the list order: the first author is slot 1, the last author is the final slot
  (OpenAlex needed `author_position_int` plus an id tie-break);
- publications with an empty `authors[]` are absent, exactly as in `paper_author.parquet`.

As in OpenAlex, an author with two countries contributes to **both** entries of
`country_author_counts`, so that column can sum to more than `n_located`; Namibia is `NA` (DuckDB
keeps it — pandas `read_csv` would not).

## Output — `Dimensions/output/paper_author_country.parquet`
| column | |
|---|---|
| `paper_id` | `pub.…`, keyed like every sibling output here |
| `team_size` | author slots (identical to `paper_author.team_size`) |
| `n_located` | slots with at least one country |
| `countries` | distinct ISO2 codes over all authors, **sorted**, `;`-joined; null if no author is located (identical to `paper_author.countries`) |
| `n_countries` | `len(countries.split(';'))`, 0 when null |
| `country_author_counts` | located authors per country, `US:3;CN:1`, ordered by count desc then code |
| `first_author_country` / `last_author_country` | the country list of the first / last slot, `;`-joined; null if *that* slot is unlocated (not promoted to the next located one) |
| `is_international` | `n_countries > 1` |

Sorted by `paper_id`. One DuckDB pass over the dump: every column is computed per row with list
functions, so the only heavy operator is the final sort, which spills. Smoke test:
`NB_DIM_BASE=<scratch> NB_FILE_LIMIT=12` (a strided sample of dump files, output under the scratch base).

In [1]:
import os, sys, time
import numpy as np, pandas as pd
import duckdb
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Dimensions')
import dim_common as dim

FILES  = dim.pub_files()                         # strided sample when NB_FILE_LIMIT is set
SMOKE  = bool(dim.FILE_LIMIT)
OUT_FP = f'{dim.OUT}/paper_author_country.parquet'
PA_FP  = f'{dim.HERE}/output/paper_author.parquet'     # sibling, cross-check (the real one, also in a smoke test)
META_FP = f'{dim.HERE}/output/paper_metadata.parquet'  # sibling, year for the coverage table only
os.makedirs(dim.OUT, exist_ok=True)
os.makedirs(dim.CACHE, exist_ok=True)

MEM = os.environ.get('NB_DUCKDB_MEM', '150GB')
con = duckdb.connect()
con.execute(f"SET memory_limit='{MEM}'")
con.execute(f"SET threads={dim.N_WORKERS}")      # DuckDB would otherwise size itself to the whole node
con.execute(f"SET temp_directory='{dim.CACHE}/duckdb_tmp_author_country'")
con.execute('SET preserve_insertion_order=false')
con.execute('SET enable_progress_bar=false')
print(f'source : {len(FILES):,} publications files under {dim.PUBS}' + ('   *** SMOKE TEST' if SMOKE else ''))
print(f'output : {OUT_FP}')
print(f'duckdb : memory_limit {MEM}, threads {dim.N_WORKERS}')

source : 4,219 publications files under /project/jevans/dimensions/dimensions/dimensions_june_2025/publications
output : /project/jevans/Dawoon/Science of Science/Dimensions/output/paper_author_country.parquet
duckdb : memory_limit 150GB, threads 8


## 1. Build

`al` is the per-slot list of country lists (`[]` for an unlocated slot), so `len(l) > 0` is the
"located" test, `al[1]` / `al[len(al)]` are the first / last author, and `cc` is the paper's
distinct, sorted union. `country_author_counts` sorts `{-count, code}` structs, i.e. count desc then
code. The dump files have one schema (checked over all 4,219 footers on 2026-10-01), so they are
read as one relation.

In [2]:
%%time
t0 = time.time()
FL = '[' + ', '.join(f"'{f}'" for f in FILES) + ']'
con.execute(f"""
COPY (
  WITH a AS (
    SELECT id AS paper_id, len(authors) AS team_size,
           list_transform(authors, x -> list_sort(list_distinct(list_filter(
               list_transform(coalesce(x.affiliations_address, []), ad -> ad.country_code),
               c -> (c IS NOT NULL AND c <> ''))))) AS al
    FROM read_parquet({FL})
    WHERE len(authors) > 0),
  b AS (SELECT *, list_sort(list_distinct(flatten(al))) AS cc FROM a)
  SELECT paper_id,
         team_size::INTEGER                                   AS team_size,
         len(list_filter(al, l -> len(l) > 0))::INTEGER       AS n_located,
         nullif(array_to_string(cc, ';'), '')                 AS countries,
         len(cc)::INTEGER                                     AS n_countries,
         nullif(array_to_string(list_transform(
             list_sort(list_transform(cc, c -> {{'k': -len(list_filter(al, l -> list_contains(l, c))), 'c': c}})),
             s -> s.c || ':' || (-s.k)::VARCHAR), ';'), '')  AS country_author_counts,
         nullif(array_to_string(al[1], ';'), '')              AS first_author_country,
         nullif(array_to_string(al[len(al)], ';'), '')        AS last_author_country,
         len(cc) > 1                                          AS is_international
  FROM b
  ORDER BY paper_id
) TO '{OUT_FP}.tmp' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)""")
os.replace(OUT_FP + '.tmp', OUT_FP)
n_papers = con.execute(f"SELECT count(*) FROM read_parquet('{OUT_FP}')").fetchone()[0]
print(f'WROTE {OUT_FP}  ({os.path.getsize(OUT_FP)/1e9:.2f} GB, {n_papers:,} publications with an author slot) '
      f'in {time.time()-t0:.0f}s')

WROTE /project/jevans/Dawoon/Science of Science/Dimensions/output/paper_author_country.parquet  (0.76 GB, 142,431,518 publications with an author slot) in 289s


## 2. Verification

1. `paper_id` unique; `n_located ≤ team_size`; `n_countries == len(countries.split(';'))`;
   the list is sorted and duplicate-free.
2. `country_author_counts` names exactly the codes in `countries`, and its sum is ≥ `n_located`
   (equal unless some author has two countries — that surplus is reported, not asserted).
3. `first_author_country` / `last_author_country` ⊆ `countries`; `is_international` agrees with
   `n_countries`.
4. Against `paper_author.parquet`: the same publications, and the same `team_size` **and**
   `countries` on every one of them — both are built from the same `authors[]`, so any difference is
   a defect. In a smoke test only the sampled publications exist here, so `paper_author`-only rows
   are expected and not counted as a failure.
5. Coverage: located share by team size and by publication decade (via `paper_metadata.year`), and
   the most frequent countries.

Checks 1–4 are asserted after the counts are printed.

In [3]:
%%time
chk = con.execute(f"""
WITH s AS (SELECT * FROM read_parquet('{OUT_FP}')),
     x AS (SELECT *, CASE WHEN countries IS NULL THEN [] ELSE str_split(countries, ';') END AS parts,
                     CASE WHEN country_author_counts IS NULL THEN []
                          ELSE list_transform(str_split(country_author_counts, ';'), z -> str_split(z, ':')) END AS cc,
                     CASE WHEN first_author_country IS NULL THEN [] ELSE str_split(first_author_country, ';') END AS fparts,
                     CASE WHEN last_author_country  IS NULL THEN [] ELSE str_split(last_author_country,  ';') END AS lparts
           FROM s)
SELECT count(*)                                                                  AS papers,
       count(*) - count(DISTINCT paper_id)                                       AS dup_paper_id,
       sum(team_size)                                                            AS author_slots,
       sum(n_located)                                                            AS located_slots,
       count(*) FILTER (WHERE n_located > team_size)                             AS located_gt_team,
       count(*) FILTER (WHERE n_countries <> len(parts))                         AS n_countries_mismatch,
       count(*) FILTER (WHERE len(parts) <> len(list_distinct(parts)))           AS dup_in_countries,
       count(*) FILTER (WHERE parts <> list_sort(parts))                         AS unsorted_countries,
       count(*) FILTER (WHERE list_sort(list_transform(cc, z -> z[1])) <> parts)  AS counts_codes_mismatch,
       count(*) FILTER (WHERE coalesce(list_sum(list_transform(cc, z -> z[2]::INTEGER)), 0) < n_located) AS counts_sum_lt_located,
       count(*) FILTER (WHERE coalesce(list_sum(list_transform(cc, z -> z[2]::INTEGER)), 0) > n_located) AS multi_country_authors_papers,
       count(*) FILTER (WHERE NOT list_has_all(parts, fparts))                   AS first_not_in_list,
       count(*) FILTER (WHERE NOT list_has_all(parts, lparts))                   AS last_not_in_list,
       count(*) FILTER (WHERE is_international <> (n_countries > 1))             AS intl_mismatch,
       count(*) FILTER (WHERE countries IS NOT NULL)                             AS with_country,
       count(*) FILTER (WHERE is_international)                                  AS international
FROM x""").fetchdf().iloc[0]
for k, v in chk.items():
    print(f'  {k:<30} {int(v):>15,}')
HARD = ['dup_paper_id', 'located_gt_team', 'n_countries_mismatch', 'dup_in_countries', 'unsorted_countries',
        'counts_codes_mismatch', 'counts_sum_lt_located', 'first_not_in_list', 'last_not_in_list', 'intl_mismatch']
ok = all(chk[k] == 0 for k in HARD)
print(f"\n1-3. {'ALL PASS' if ok else 'FAILED -- see the counts above'}")
print(f'     located share: {chk.with_country/chk.papers*100:.1f}% of publications, '
      f'{chk.located_slots/chk.author_slots*100:.1f}% of author slots; international: {chk.international/chk.papers*100:.1f}%; '
      f'publications with a multi-country author: {int(chk.multi_country_authors_papers):,}')

m = con.execute(f"""
WITH n AS (SELECT paper_id, team_size, countries FROM read_parquet('{OUT_FP}')),
     o AS (SELECT paper_id, team_size, countries FROM read_parquet('{PA_FP}'))
SELECT (SELECT count(*) FROM n JOIN o USING (paper_id))                                          AS shared,
       (SELECT count(*) FROM n JOIN o USING (paper_id) WHERE n.team_size <> o.team_size)          AS team_size_mismatch,
       (SELECT count(*) FROM n JOIN o USING (paper_id) WHERE n.countries IS DISTINCT FROM o.countries) AS countries_mismatch,
       (SELECT count(*) FROM n ANTI JOIN o USING (paper_id))                                     AS new_only,
       (SELECT count(*) FROM o ANTI JOIN n USING (paper_id))                                     AS old_only""").fetchdf().iloc[0]
for k, v in m.items():
    print(f'  {k:<30} {int(v):>15,}')
ok4 = m.team_size_mismatch == 0 and m.countries_mismatch == 0 and m.new_only == 0 and (SMOKE or m.old_only == 0)
print(f"4.   vs paper_author.parquet: {'OK' if ok4 else 'FAILED'}" + ('  (smoke: paper_author-only rows expected)' if SMOKE else ''))

print('\n5.   coverage by team size')
display(con.execute(f"""SELECT team_size, count(*) AS papers,
                        round(100.0*count(countries)/count(*), 1) AS pct_located,
                        round(100.0*count(*) FILTER (WHERE is_international)/count(*), 2) AS pct_international
                        FROM read_parquet('{OUT_FP}') GROUP BY 1 ORDER BY 1 LIMIT 12""").fetchdf())
print('     coverage by publication decade (paper_metadata.year)')
display(con.execute(f"""
SELECT (m.year // 10) * 10 AS decade, count(*) AS papers,
       round(100.0*count(s.countries)/count(*), 1) AS pct_located,
       round(100.0*count(*) FILTER (WHERE s.is_international)/count(*), 2) AS pct_international
FROM read_parquet('{OUT_FP}') s JOIN read_parquet('{META_FP}') m USING (paper_id)
WHERE m.year BETWEEN 1900 AND 2025 GROUP BY 1 ORDER BY 1""").fetchdf())
print('     most frequent countries (publications with >=1 author there)')
display(con.execute(f"""
SELECT c AS country, count(*) AS papers, round(100.0*count(*)/(SELECT count(*) FROM read_parquet('{OUT_FP}')), 2) AS pct
FROM read_parquet('{OUT_FP}'), unnest(str_split(countries, ';')) AS u(c)
WHERE countries IS NOT NULL GROUP BY 1 ORDER BY 2 DESC LIMIT 15""").fetchdf())
display(con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') WHERE is_international ORDER BY n_countries DESC LIMIT 5").fetchdf())
con.close()
assert ok, 'internal consistency checks 1-3 failed'
assert ok4, 'cross-check against paper_author.parquet failed'
print('\nALL CHECKS PASSED')

  papers                             142,431,518
  dup_paper_id                                 0
  author_slots                       496,565,750
  located_slots                      343,384,197
  located_gt_team                              0
  n_countries_mismatch                         0
  dup_in_countries                             0
  unsorted_countries                           0
  counts_codes_mismatch                        0
  counts_sum_lt_located                        0
  multi_country_authors_papers         4,065,345
  first_not_in_list                            0
  last_not_in_list                             0
  intl_mismatch                                0
  with_country                        86,405,175
  international                       13,556,134

1-3. ALL PASS
     located share: 60.7% of publications, 69.2% of author slots; international: 9.5%; publications with a multi-country author: 4,065,345
  shared                             142,431,518
  team_size_m

,team_size,papers,pct_located,pct_international
0,1,45081946,36.3,0.40
1,2,26652040,60.2,6.06
2,3,21121722,68.7,10.16
3,4,15266344,74.7,13.82
4,5,10738645,78.0,16.46
5,6,7626040,81.1,18.65
6,7,4849815,83.3,21.35
7,8,3333340,84.3,23.37
8,9,2199197,85.9,26.04
9,10,1639512,85.2,27.11


     coverage by publication decade (paper_metadata.year)


,decade,papers,pct_located,pct_international
0,1900,387566,16.3,0.37
1,1910,441688,18.7,0.40
2,1920,647828,25.3,0.46
3,1930,904238,28.8,0.55
4,1940,997420,24.6,0.28
5,1950,2309565,25.4,0.34
6,1960,3990385,33.6,0.83
7,1970,6336513,40.8,1.64
8,1980,9203243,50.9,3.06
9,1990,13643347,65.6,5.76


     most frequent countries (publications with >=1 author there)


,country,papers,pct
0,US,24574599,17.25
1,CN,10755942,7.55
2,GB,6965535,4.89
3,JP,6207652,4.36
4,DE,5858287,4.11
5,IN,3659560,2.57
6,FR,3542475,2.49
7,CA,3354146,2.35
8,IT,3052881,2.14
9,AU,2551523,1.79


,paper_id,team_size,n_located,countries,n_countries,country_author_counts,first_author_country,last_author_country,is_international
0,pub.1125118184,473,472,AF;AL;AM;AO;AR;AT;AU;AZ;BD;BE;BF;BG;BI;BJ;BO;B...,129,US:48;IN:26;CN:17;ID:16;NG:13;TH:13;GB:12;PK:1...,None,GB,True
1,pub.1156597783,1626,1626,AE;AL;AM;AO;AR;AT;AU;AZ;BA;BB;BD;BE;BG;BJ;BN;B...,128,GB:1626;IR:92;BR:85;IT:65;CN:64;ES:61;DE:51;DK...,GB,GB;GH,True
2,pub.1136234646,1381,1381,AE;AL;AR;AT;AU;BA;BB;BD;BE;BG;BJ;BN;BR;CA;CH;C...,116,IR:84;BR:65;GB:60;IT:59;CN:58;ES:50;IN:47;DK:4...,GB,GB,True
3,pub.1113952920,1155,1155,AE;AL;AR;AT;AU;BA;BB;BD;BE;BG;BJ;BN;BR;CA;CH;C...,111,GB:1155;IT:54;BR:47;CN:47;IR:47;ES:43;IN:34;NL...,GB,GB,True
4,pub.1188406028,1447,1447,AE;AG;AL;AR;AT;AU;BD;BE;BG;BH;BJ;BN;BO;BR;BW;C...,104,PE:1447;IR:221;US:177;IN:154;AU:110;ET:95;GB:6...,PE,PE;PH,True



ALL CHECKS PASSED
